# Notebook 1A — Inference via OpenRouter API
**MedThink-Bench VI Eval Pipeline**

Output: `checkpoint_inference.jsonl` → unified JSONL schema (dùng chung với Notebook 1B)

---
**Yêu cầu**: `pip install aiohttp`

In [1]:
# ── Cell 1: Install dependencies ────────────────────────────────────────
!pip install aiohttp -q

In [2]:
# ── Cell 1.5: Copy shared modules từ dataset vào working dir ────────────
import shutil, os

_SRC = "/kaggle/input/datasets/quangminh2401/medthink-vi-eval-pipeline"
_DST = "/kaggle/working"

for _f in ["utils.py", "async_openrouter.py", "evaluator.py"]:
    shutil.copy(os.path.join(_SRC, _f), os.path.join(_DST, _f))
    print(f"copied: {_f}")


copied: utils.py
copied: async_openrouter.py
copied: evaluator.py


In [3]:
# ── Cell 2: CONFIG - Điền thông tin cấu hình tại đây ───────────────────────────────────────────────────────
import sys
sys.path.insert(0, "/kaggle/working")

# Load API key từ Kaggle Secrets (Settings → Secrets → OPENROUTER_API_KEY)
from kaggle_secrets import UserSecretsClient
OPENROUTER_API_KEY = UserSecretsClient().get_secret("OPENROUTER_API_KEY")

# ── Đường dẫn file ──────────────────────────
DATASET_PATH        = "/kaggle/input/datasets/quangminh2401/medthink-benchfull-translated/vi_QA_data.jsonl"   # Dataset tiếng Việt
CHECKPOINT_INFER    = "/kaggle/working/checkpoint_inference.jsonl"    # Resume inference

# ── OpenRouter API ───────────────────────────
OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1/chat/completions"

# ── Model inference (Notebook 1A – OpenRouter) ──
INFER_API_MODEL     = "anthropic/claude-sonnet-4.6"                    # Model để test
INFER_MAX_TOKENS    = 4096
INFER_TEMPERATURE   = 0.0
INFER_SEMAPHORE     = 8          # Số concurrent requests tối đa
INFER_RETRY_MAX     = 3
INFER_TIMEOUT_SEC   = 180

# ── Prompt inference ─────────────────────────
INFER_SYSTEM_PROMPT = """Bạn là một bác sĩ chuyên khoa đang trả lời câu hỏi trắc nghiệm y khoa.
Hãy suy luận từng bước, sau đó đưa ra đáp án cuối cùng.

Trả lời ĐÚNG theo định dạng JSON sau, không thêm bất kỳ nội dung nào khác:
{"answer": "<đáp án trắc nghiệm>", "reasoning": "<lý luận từng bước của bạn>"}"""

INFER_USER_TEMPLATE = """{question}"""

print(f"Model   : {INFER_API_MODEL}")
print(f"Dataset : {DATASET_PATH}")
print(f"Output  : {CHECKPOINT_INFER}")

Model   : anthropic/claude-sonnet-4.6
Dataset : /kaggle/input/datasets/quangminh2401/medthink-benchfull-translated/vi_QA_data.jsonl
Output  : /kaggle/working/checkpoint_inference.jsonl


In [4]:
"""
# ── Copy checkpoint từ dataset vào working dir để resume ────────────────
import shutil, os

CHECKPOINT_SRC = "/kaggle/working/checkpoint_inference.jsonl"
CHECKPOINT_DST = "/kaggle/working/checkpoint_inference.jsonl"

if not os.path.exists(CHECKPOINT_DST):
    shutil.copy(CHECKPOINT_SRC, CHECKPOINT_DST)
    print(f"Copied checkpoint: {CHECKPOINT_DST}")
else:
    print("Checkpoint đã tồn tại trong working dir, giữ nguyên.")

"""

'\n# ── Copy checkpoint từ dataset vào working dir để resume ────────────────\nimport shutil, os\n\nCHECKPOINT_SRC = "/kaggle/working/checkpoint_inference.jsonl"\nCHECKPOINT_DST = "/kaggle/working/checkpoint_inference.jsonl"\n\nif not os.path.exists(CHECKPOINT_DST):\n    shutil.copy(CHECKPOINT_SRC, CHECKPOINT_DST)\n    print(f"Copied checkpoint: {CHECKPOINT_DST}")\nelse:\n    print("Checkpoint đã tồn tại trong working dir, giữ nguyên.")\n\n'

In [5]:
# ── Cell 3: Load dataset + resume checkpoint ────────────────────────────
import json
from utils import load_jsonl, load_checkpoint_indices, TokenTracker

dataset = load_jsonl(DATASET_PATH)
done_indices = load_checkpoint_indices(CHECKPOINT_INFER)
todo = [
    {**s, "_original_position": i}
    for i, s in enumerate(dataset)
    if s["Index"] not in done_indices
]

print(f"Tổng samples   : {len(dataset)}")
print(f"Đã xử lý       : {len(done_indices)}")
print(f"Còn lại        : {len(todo)}")

Tổng samples   : 500
Đã xử lý       : 0
Còn lại        : 500


In [6]:
# ── Cell 4: Chạy inference async ────────────────────────────────────────
import asyncio
from tqdm.notebook import tqdm

from async_openrouter import AsyncOpenRouterClient
from utils import (
    append_jsonl,
    extract_answer_from_json,
    extract_reasoning_from_json,
    TokenTracker,
)

tracker = TokenTracker(label="Inference API")


def build_messages(sample: dict) -> list[dict]:
    return [
        {"role": "system", "content": INFER_SYSTEM_PROMPT},
        {"role": "user",   "content": INFER_USER_TEMPLATE.format(
            question=sample["question"]
        )},
    ]


async def process_sample(client, sample, pbar):
    messages = build_messages(sample)
    raw_output = await client.chat(messages)

    if raw_output is None:
        # Không ghi vào checkpoint — resume sẽ tự retry
        pbar.update(1)
        return

    extracted_answer = extract_answer_from_json(raw_output)
    reasoning = extract_reasoning_from_json(raw_output)

    record = {
        "_original_position": sample["_original_position"],  # giữ để sort
        "index"             : sample["Index"],
        "qa_type"           : sample.get("QA_Type", ""),
        "question"          : sample["question"],
        "answer"            : sample["answer"],
        "scoring_points"    : sample.get("Scoring_Points", []),
        "raw_output"        : raw_output,
        "extracted_answer"  : extracted_answer,
        "reasoning"         : reasoning,
        "model_id"          : INFER_API_MODEL,
        "inference_backend" : "openrouter",
    }
    append_jsonl(CHECKPOINT_INFER, record)
    pbar.update(1)


async def run_all():
    async with AsyncOpenRouterClient(
        api_key        = OPENROUTER_API_KEY,
        base_url       = OPENROUTER_BASE_URL,
        model          = INFER_API_MODEL,
        max_tokens     = INFER_MAX_TOKENS,
        temperature    = INFER_TEMPERATURE,
        semaphore_limit= INFER_SEMAPHORE,
        retry_max      = INFER_RETRY_MAX,
        timeout_sec    = INFER_TIMEOUT_SEC,
        tracker        = tracker,
    ) as client:
        with tqdm(total=len(todo), desc="Inference") as pbar:
            tasks = [process_sample(client, s, pbar) for s in todo]
            await asyncio.gather(*tasks)


await run_all()

# ── In tổng token sau khi cell chạy xong ──
tracker.print_summary()

Inference:   0%|          | 0/500 [00:00<?, ?it/s]


  [Inference API] Token Usage Summary
  API calls       : 500
  Prompt tokens   : 329,265
  Completion tokens: 432,902
  Total tokens    : 762,167
  Elapsed time    : 999.7s
  Avg tokens/call : 1,524



In [7]:
# ── Cell 5: Kiểm tra kết quả ────────────────────────────────────────────
from utils import load_jsonl

results = load_jsonl(CHECKPOINT_INFER)
done_indices = {r["index"] for r in results}
incomplete = [s for s in dataset if s["Index"] not in done_indices]

n_total  = len(results)
n_parsed = sum(1 for r in results if r.get("extracted_answer") is not None)

print(f"Hoàn thành     : {n_total} / {len(dataset)}")
print(f"Chưa xử lý    : {len(incomplete)} samples: {[s['Index'] for s in incomplete]}")
print(f"Parse được A/B/C/D: {n_parsed} ({n_parsed/n_total*100:.1f}%)")
print()
print("Sample đầu tiên:")
print(json.dumps(results[0], ensure_ascii=False, indent=2))

Hoàn thành     : 500 / 500
Chưa xử lý    : 0 samples: []
Parse được A/B/C/D: 500 (100.0%)

Sample đầu tiên:
{
  "_original_position": 7,
  "index": 8,
  "qa_type": "Thu thập thông tin bệnh nhân và Đánh giá chẩn đoán",
  "question": "Một người đàn ông 20 tuổi bị nhiễm trùng đường hô hấp trên thường xuyên trong 4 năm qua. Bệnh nhân có đờm mủ hằng ngày và nhận thấy khả năng gắng sức giảm trong 2 năm qua. Bệnh nhân và vợ không thể thụ thai do số lượng tinh trùng thấp. Nghe thấy tiếng khò khè thì thở ra rải rác và ran ngáy khắp cả hai phế trường. X-quang ngực cho thấy ứ khí phổi. Đo hô hấp ký cho thấy tỷ lệ FEV1:FVC giảm. Điều nào sau đây có khả năng xác nhận chẩn đoán nhất?\nA. Nội soi phế quản\nB. Xét nghiệm đờm tìm bạch cầu ái toan\nC. Tế bào học đờm\nD. Xét nghiệm chloride trong mồ hôi\nE. Chụp CT ngực\nF. Siêu âm tim\nG. Đo hô hấp ký lặp lại\nH. Phản ứng da tuberculin\nI. Công thức máu toàn bộ\nJ. Phân tích khí máu động mạch",
  "answer": "D. Xét nghiệm chloride trong mồ hôi",
  "scori

## **DEBUG SECTION**

In [8]:
# ── Cell DEBUG: retry 1 sample + xem full HTTP response ─────────────────
# Điền index của sample muốn debug vào đây
DEBUG_INDEX = todo[0]

import aiohttp, asyncio, json as _json

debug_sample = next((s for s in dataset if s["Index"] == DEBUG_INDEX), None)
if debug_sample is None:
    print(f"Không tìm thấy sample index {DEBUG_INDEX}")
else:
    async def debug_single():
        messages = [
            {"role": "system", "content": INFER_SYSTEM_PROMPT},
            {"role": "user",   "content": INFER_USER_TEMPLATE.format(
                question=debug_sample["question"]
            )},
        ]
        payload = {
            "model"      : INFER_API_MODEL,
            "messages"   : messages,
            "max_tokens" : INFER_MAX_TOKENS,
            "temperature": INFER_TEMPERATURE,
        }
        headers = {
            "Authorization": f"Bearer {OPENROUTER_API_KEY}",
            "Content-Type" : "application/json",
            "HTTP-Referer" : "https://kaggle.com",
        }
        timeout = aiohttp.ClientTimeout(total=120)
        async with aiohttp.ClientSession(headers=headers, timeout=timeout) as session:
            async with session.post(OPENROUTER_BASE_URL, json=payload) as resp:
                status = resp.status
                body   = await resp.text()

        print(f"HTTP Status : {status}")
        print(f"{'='*60}")
        try:
            parsed = _json.loads(body)
            print(_json.dumps(parsed, ensure_ascii=False, indent=2))
        except Exception:
            print(body)

    #await debug_single()

Không tìm thấy sample index {'Index': 1, 'QA_Type': 'Thu thập thông tin bệnh nhân và Đánh giá chẩn đoán', 'question': 'Một nam giới 50 tuổi đến khám với trán vồ, xương mũi to, hàm to và các ngón tay hình xẻng. Bạn sẽ chỉ định xét nghiệm nào sau đây để chẩn đoán?\nA. IGF1\nB. ACTH\nC. TSH\nD. Cortisol huyết thanh', 'answer': 'A. IGF1', 'Scoring_Points': ['Trán vồ, chứng nhô hàm (tiền hàm nhô) và các ngón tay hình xẻng là những đặc điểm điển hình của bệnh to đầu chi (acromegaly) do dư thừa hormone tăng trưởng ở người lớn.', 'Nồng độ yếu tố tăng trưởng giống insulin-1 (IGF-1) phản ánh chính xác mức bài tiết hormone tăng trưởng trung bình và là xét nghiệm chẩn đoán ban đầu được khuyến cáo khi nghi ngờ bệnh to đầu chi.'], '_original_position': 0}


## **EXPORT SECTION**

In [9]:
# ── Cell xuất file cuối: sort + strip _original_position ────────────────
import json, shutil

results = load_jsonl(CHECKPOINT_INFER)

# Sort theo thứ tự dataset gốc
results_sorted = sorted(results, key=lambda r: r.get("_original_position", 0))

# Strip _original_position trước khi xuất
OUTPUT_INFER = "/kaggle/working/inference_results.jsonl"
with open(OUTPUT_INFER, "w", encoding="utf-8") as f:
    for r in results_sorted:
        clean = {k: v for k, v in r.items() if k != "_original_position"}
        f.write(json.dumps(clean, ensure_ascii=False) + "\n")

print(f"Đã xuất {len(results_sorted)} records → {OUTPUT_INFER}")
print("Thứ tự đầu tiên:")
for r in results_sorted[:3]:
    print(f"  position={r['_original_position']}  index={r['index']}")

Đã xuất 500 records → /kaggle/working/inference_results.jsonl
Thứ tự đầu tiên:
  position=0  index=1
  position=1  index=2
  position=2  index=3
